In [ ]:
import os, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import snowballstemmer
from nltk.corpus import stopwords

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
import matplotlib.cm as cm

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from keras.layers import LSTM
from tensorflow.keras.layers import Input, Dense, Embedding, SpatialDropout1D, add, concatenate

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import nltk
from nltk import PorterStemmer

In [ ]:
!pip install keras

In [ ]:
!pip install snowballstemmer

In [ ]:
!pip install matplotlib

In [ ]:
!pip install seaborn

In [ ]:
df = pd.read_csv('datasets/labelled_airline_tweet.csv')
df.head()

In [ ]:
data = df[['topic_desc', 'text']].copy()
data.head()

In [ ]:
pd.set_option('display.max_colwidth', 1)
data.sample(5)

In [ ]:
data['topic_desc'].value_counts()

In [ ]:
def print_plot(index):
    example = df[df.index == index][['text','topic_desc']].values[0]
    if len(example) > 0 and example is not None:
        print(example[0])
        print('Greivance type:', example[1])
print_plot(10)

In [ ]:
#Text Preprocessing
import re

def clean_text(txt):
    
    """
    removing all hashtags , punctuations, stop_words  and links, also stemming words 
    """
    txt = txt.lower()
    txt = re.sub(r"(?<=\w)nt", "not",txt) #change don't to do not cna't to cannot 
    txt = re.sub(r"(@\S+)", "", txt)  # remove hashtags
    txt = re.sub(r'\W', ' ', str(txt)) # remove all special characters including apastrophie 
    txt = txt.translate(str.maketrans('', '', string.punctuation)) # remove punctuations 
    txt = re.sub(r'\s+[a-zA-Z]\s+', ' ', txt)   # remove all single characters (it's -> it s then we need to remove s)
    txt = re.sub(r'\s+', ' ', txt, flags=re.I) # Substituting multiple spaces with single space
    txt = re.sub(r"(http\S+|http)", "", txt) # remove links 
    txt = ' '.join([PorterStemmer().stem(word=word) for word in txt.split(" ") if word not in stopwords.words('english') ]) # stem & remove stop words
    txt = ''.join([i for i in txt if not i.isdigit()]).strip() # remove digits ()
    return txt


In [ ]:
print('Original Text : ',data['text'][5])  
print('Processed Text : ',clean_text(data['text'][5]))

In [ ]:
data['text'] = data['text'].apply(clean_text)
data['text'].values

In [ ]:
MAX_NB_WORDS = 50000
MAX_SEQUENCE_LENGTH = 250
EMBEDDING_DIM = 100
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
tokenizer.fit_on_texts(df['text'].values)
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

In [ ]:
# Word Embedding
X = tokenizer.texts_to_sequences(df['text'].values)
X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)
print('Shape of data tensor:', X.shape[1])

In [ ]:
#Converting categorical labels to numbers.
Y = pd.get_dummies(df['topic_desc']).values
print('Shape of label tensor:', Y.shape)

In [ ]:
#Train-Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.10, random_state = 42)
print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)

In [ ]:
# Model Structuring
model = Sequential()
model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(8, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

epochs = 10
batch_size = 64
early_stopping = EarlyStopping(monitor='val_loss', patience=20, verbose=1, mode='auto')

history = model.fit(X_train, Y_train, epochs=epochs, batch_size=batch_size,validation_split=0.1,callbacks=[early_stopping])

In [ ]:
model.summary()

In [ ]:
import pickle

model.save_weights("models/multiclassComplaintClassifier.h5")

with open("models/tokenizerMulticlassComplaintClassification.pickle", 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
#Accuracy
acr = model.evaluate(X_test,Y_test)
print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(acr[0],acr[1]))

In [ ]:
#Accuracy Plot
plt.title('Accuracy')
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='test')
plt.legend()
plt.show()

In [ ]:
# Test a new complaint
new_tweet = ['is todays flight from Palm Springs Ca to JFK in NY Cancelled Flightled']
seq = tokenizer.texts_to_sequences(new_tweet)
padded = pad_sequences(seq, maxlen=MAX_SEQUENCE_LENGTH)
pred = model.predict(padded)
labels = ['Reschedule and Refund', 'Baggage Issue','Phone and Online Booking', 'Extra Charges', 'Delay and Customer Service', 'Seating Preferences', 'Reservation Issue', 'Customer Experience']
print(pred, labels[np.argmax(pred)])

In [ ]:
class MulticlassComplainInference:

  def __init__(self):
      self.load_models()

  def get_model(self):
      model = Sequential()
      model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
      model.add(SpatialDropout1D(0.2))
      model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
      model.add(Dense(8, activation='softmax'))
      model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
      return model
  
  
  def load_models(self):
      with open('models/tokenizerMulticlassComplaintClassification.pickle', 'rb') as handle:
          self.tokenizer = pickle.load(handle)

      self.model = self.get_model()
      self.model.load_weights("models/multiclassComplaintClassifier.h5")

  def clean_text(self, txt):
    
      """
      removing all hashtags , punctuations, stop_words  and links, also stemming words 
      """
      txt = txt.lower()
      txt = re.sub(r"(?<=\w)nt", "not",txt) #change don't to do not cna't to cannot 
      txt = re.sub(r"(@\S+)", "", txt)  # remove hashtags
      txt = re.sub(r'\W', ' ', str(txt)) # remove all special characters including apastrophie 
      txt = txt.translate(str.maketrans('', '', string.punctuation)) # remove punctuations 
      txt = re.sub(r'\s+[a-zA-Z]\s+', ' ', txt)   # remove all single characters (it's -> it s then we need to remove s)
      txt = re.sub(r'\s+', ' ', txt, flags=re.I) # Substituting multiple spaces with single space
      txt = re.sub(r"(http\S+|http)", "", txt) # remove links 
      txt = ' '.join([PorterStemmer().stem(word=word) for word in txt.split(" ") if word not in stopwords.words('english') ]) # stem & remove stop words
      txt = ''.join([i for i in txt if not i.isdigit()]).strip() # remove digits ()
      return txt
  
  def predict_complaint(self, text):
      new_tweet = self.clean_text(text)
      seq = self.tokenizer.texts_to_sequences([new_tweet])
      padded = pad_sequences(seq, maxlen=MAX_SEQUENCE_LENGTH)
      pred = self.model.predict(padded)
      labels = ['Reschedule and Refund', 'Baggage Issue','Phone and Online Booking', 'Extra Charges', 
                'Delay and Customer Service', 'Seating Preferences', 'Reservation Issue', 'Customer Experience']
      print(pred, labels[np.argmax(pred)])
      return labels[np.argmax(pred)]


In [ ]:
mul = MulticlassComplainInference()

In [ ]:
mul.predict_complaint("@united I have never been mislead by a company as many times as I have this week by United Airlines!")